# 2026 COMP90042 Project: Single-Notebook Fact Checking Pipeline

This notebook is a self-contained Colab pipeline for candidate generation, top3 evidence reranking, and claim classification.
It trains all task-specific models during the notebook run and does not require a saved project checkpoint.

# Readme

Expected files in `DATA_DIR`:

- `train-claims.json`
- `dev-claims.json` or `test-claims-unlabelled.json`
- `evidence.json`

The main path is:

1. query-restricted BM25 candidate generation on CPU;
2. BGE-small dense reranking inside the sparse candidate pool on GPU when available;
3. MiniLM cross-encoder binary evidence selector for top3;
4. DistilRoBERTa concat classifier for the final claim label.

The notebook prints GPU information with `nvidia-smi` and writes final predictions to `OUTPUT_DIR/final_predictions.json`.

# 1.DataSet Processing

In [1]:
# Colab dependency setup.
# Keep dependencies open-source and lightweight enough for a standard Colab runtime.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "sentencepiece": "sentencepiece",
    "safetensors": "safetensors",
}
missing = [pkg for mod, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
# Runtime configuration.
from pathlib import Path
import os


def find_local_project_root():
    current = Path.cwd().resolve()
    for root in [current, *current.parents]:
        if (root / 'data' / 'evidence.json').exists():
            return root
    return None


# Auto-detect Colab vs local repo.
# In Colab, put the course JSON files directly under /content, or set DATA_DIR manually.
# Locally, this falls back to the repo data/ directory.
LOCAL_PROJECT_ROOT = find_local_project_root()
COLAB_ROOT = Path('/content')
RUNNING_IN_COLAB = COLAB_ROOT.exists() and os.access(COLAB_ROOT, os.W_OK)

if RUNNING_IN_COLAB:
    DATA_DIR = COLAB_ROOT
    OUTPUT_DIR = COLAB_ROOT / 'a3_full_pipeline_outputs'
elif LOCAL_PROJECT_ROOT is not None:
    DATA_DIR = LOCAL_PROJECT_ROOT / 'data'
    OUTPUT_DIR = LOCAL_PROJECT_ROOT / 'outputs' / 'round16_colab_full_pipeline_local'
else:
    DATA_DIR = Path.cwd()
    OUTPUT_DIR = Path.cwd() / 'a3_full_pipeline_outputs'

# Manual override examples:
# DATA_DIR = Path('/content/drive/MyDrive/COMP90042_A3/data')
# OUTPUT_DIR = Path('/content/drive/MyDrive/COMP90042_A3_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CLAIMS_PATH = DATA_DIR / 'train-claims.json'
TARGET_CLAIMS_PATH = DATA_DIR / 'dev-claims.json'  # change to test-claims-unlabelled.json for final prediction
EVIDENCE_PATH = DATA_DIR / 'evidence.json'

missing_files = [p for p in [TRAIN_CLAIMS_PATH, TARGET_CLAIMS_PATH, EVIDENCE_PATH] if not p.exists()]
if missing_files:
    raise FileNotFoundError('Missing required data files: ' + ', '.join(str(p) for p in missing_files))

SEED = 13
SMOKE_MODE = False  # set True only for a quick syntax/runtime check
SMOKE_TRAIN_CLAIMS = 80
SMOKE_TARGET_CLAIMS = 20
SMOKE_EVIDENCE_LIMIT = 50000

# Candidate generation and reranking knobs.
BM25_TOP_K = 4500
DENSE_PREFILTER_K = 2000
DENSE_TOP_K = 200
BINARY_SCORE_TOP_K = 80
FINAL_EVIDENCE_TOP_K = 3
CLASSIFIER_CONTEXT_TOP_K = 10

# Models. These are downloaded from Hugging Face at runtime, not saved project checkpoints.
DENSE_MODEL_NAME = 'BAAI/bge-small-en-v1.5'
BINARY_SELECTOR_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L6-v2'
CLAIM_CLASSIFIER_MODEL_NAME = 'distilroberta-base'

# Training knobs.
BINARY_TRAIN_SOURCE_K = 30
BINARY_MAX_NEGATIVES = 60
BINARY_EPOCHS = 2
BINARY_BATCH_SIZE = 24
BINARY_MAX_LENGTH = 256

CLASSIFIER_EPOCHS = 5
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_MAX_LENGTH = 512
EVIDENCE_TOKEN_BUDGET = 45

print('RUNNING_IN_COLAB =', RUNNING_IN_COLAB)
print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


RUNNING_IN_COLAB = False
DATA_DIR = /mnt/a/code/NLP/A3/data
OUTPUT_DIR = /mnt/a/code/NLP/A3/outputs/round16_colab_full_pipeline_local


In [3]:
# GPU visibility and deterministic setup.
import gc
import json
import math
import os
import random
import re
import subprocess
import time
from collections import Counter, defaultdict
from pathlib import Path

import heapq
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, set_seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('WARNING: GPU is not available. The notebook will run, but neural stages will be much slower.')

torch: 2.11.0+cu130
DEVICE: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
Tue May  5 14:01:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   49C    P0             51W /  300W |     994MiB /  16303MiB |      1%      Default |
|                                         |                        |       

In [4]:
# JSON loading and optional smoke slicing.
def load_json(path):
    path = Path(path)
    with path.open(encoding='utf-8') as f:
        return json.load(f)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def limit_items(obj, n):
    if not n or n <= 0:
        return obj
    return dict(list(obj.items())[:n])


def subset_evidence(evidence, train_claims, target_claims, limit):
    if not limit or limit <= 0 or limit >= len(evidence):
        return evidence
    keep = set(list(evidence)[:limit])
    for claims in (train_claims, target_claims):
        for claim in claims.values():
            for eid in claim.get('evidences', []):
                keep.add(eid)
    return {eid: evidence[eid] for eid in evidence if eid in keep}

train_claims = load_json(TRAIN_CLAIMS_PATH)
target_claims = load_json(TARGET_CLAIMS_PATH)
evidence = load_json(EVIDENCE_PATH)

if SMOKE_MODE:
    train_claims = limit_items(train_claims, SMOKE_TRAIN_CLAIMS)
    target_claims = limit_items(target_claims, SMOKE_TARGET_CLAIMS)
    evidence = subset_evidence(evidence, train_claims, target_claims, SMOKE_EVIDENCE_LIMIT)
    BM25_TOP_K = min(BM25_TOP_K, 500)
    DENSE_PREFILTER_K = min(DENSE_PREFILTER_K, 300)
    DENSE_TOP_K = min(DENSE_TOP_K, 80)
    BINARY_SCORE_TOP_K = min(BINARY_SCORE_TOP_K, 40)
    BINARY_EPOCHS = 1
    CLASSIFIER_EPOCHS = 1

print('train claims:', len(train_claims))
print('target claims:', len(target_claims))
print('evidence:', len(evidence))
print('target has labels:', all('claim_label' in c for c in target_claims.values()))

train claims: 1228
target claims: 154
evidence: 1208827
target has labels: True


In [5]:
# Evaluation helpers equivalent to the assignment metric.
LABELS = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}


def majority_label(claims):
    counts = Counter(claim.get('claim_label') for claim in claims.values() if claim.get('claim_label'))
    return counts.most_common(1)[0][0]


def predictions_from_ranked(claims, ranked, labels_by_claim=None, default_label='SUPPORTS', top_k=3):
    preds = {}
    for cid, claim in claims.items():
        label = default_label
        if labels_by_claim and cid in labels_by_claim:
            label = labels_by_claim[cid]
        elif claim.get('claim_label'):
            label = claim['claim_label']
        preds[cid] = {
            'claim_text': claim['claim_text'],
            'claim_label': label,
            'evidences': [row['evidence_id'] for row in ranked.get(cid, [])[:top_k]],
        }
    return preds


def assignment_metrics(claims, predictions):
    f_scores, acc = [], []
    for cid, claim in sorted(claims.items()):
        if cid not in predictions or 'claim_label' not in predictions[cid] or 'evidences' not in predictions[cid]:
            continue
        gold_label = claim.get('claim_label')
        if gold_label is not None:
            acc.append(float(predictions[cid]['claim_label'] == gold_label))
        gold = claim.get('evidences', [])
        pred = predictions[cid].get('evidences', [])
        if gold and pred:
            correct = len(set(gold) & set(pred))
            recall = correct / len(gold)
            precision = correct / len(pred)
            f = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        else:
            f = 0.0
        f_scores.append(f)
    mean_f = float(np.mean(f_scores)) if f_scores else 0.0
    mean_acc = float(np.mean(acc)) if acc else 0.0
    harmonic = 0.0 if mean_f + mean_acc == 0 else 2 * mean_f * mean_acc / (mean_f + mean_acc)
    return {'evidence_f': mean_f, 'accuracy': mean_acc, 'harmonic': harmonic}


def macro_recall(claims, predictions):
    recalls = []
    for cid, claim in claims.items():
        gold = claim.get('evidences', [])
        if not gold:
            continue
        pred = predictions.get(cid, {}).get('evidences', [])
        recalls.append(len(set(gold) & set(pred)) / len(gold))
    return float(np.mean(recalls)) if recalls else 0.0


def print_retrieval_metrics(name, claims, ranked, top_k=3):
    if not all('evidences' in c for c in claims.values()):
        print(name, 'target has no gold evidences; metrics skipped')
        return {}
    preds = predictions_from_ranked(claims, ranked, default_label=majority_label(train_claims), top_k=top_k)
    metrics = assignment_metrics(claims, preds)
    metrics['macro_recall'] = macro_recall(claims, preds)
    print(name, json.dumps(metrics, indent=2))
    return metrics

# 2.Model Implementation

In [6]:
# Query-restricted BM25. This avoids building a full corpus-wide BM25 object in RAM.
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
STOPWORDS = set(ENGLISH_STOP_WORDS)


def analyze_terms(text):
    tokens = [t.lower() for t in TOKEN_RE.findall(text)]
    tokens = [t for t in tokens if len(t) > 1 and t not in STOPWORDS]
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def build_query_bm25_pool(claims, evidence, top_k, k1=1.5, b=0.75, max_df_ratio=0.25):
    start = time.perf_counter()
    claim_terms = {cid: Counter(analyze_terms(c['claim_text'])) for cid, c in claims.items()}
    query_vocab = set()
    for counts in claim_terms.values():
        query_vocab.update(counts)
    print(f'BM25 query vocab: {len(query_vocab)} terms')

    evidence_ids = list(evidence.keys())
    postings = defaultdict(list)
    df = Counter()
    doc_len = np.zeros(len(evidence_ids), dtype=np.float32)

    for idx, eid in enumerate(evidence_ids):
        terms = analyze_terms(evidence[eid])
        doc_len[idx] = max(1, len(terms))
        counts = Counter(t for t in terms if t in query_vocab)
        for term, tf in counts.items():
            postings[term].append((idx, tf))
            df[term] += 1
        if idx and idx % 100000 == 0:
            print(f'BM25 index pass: {idx}/{len(evidence_ids)}')

    n_docs = len(evidence_ids)
    avgdl = float(doc_len.mean()) if len(doc_len) else 1.0
    max_df = max(1, int(max_df_ratio * n_docs))
    idf = {}
    kept_postings = {}
    for term, plist in postings.items():
        if df[term] <= max_df:
            idf[term] = math.log((n_docs - df[term] + 0.5) / (df[term] + 0.5) + 1.0)
            kept_postings[term] = plist
    print(f'BM25 postings kept: {len(kept_postings)}/{len(postings)} terms')

    pool = {}
    for i, (cid, q_counts) in enumerate(claim_terms.items(), start=1):
        scores = defaultdict(float)
        for term, qtf in q_counts.items():
            plist = kept_postings.get(term)
            if not plist:
                continue
            term_weight = idf[term] * (1.0 + 0.1 * min(qtf, 3))
            for doc_idx, tf in plist:
                denom = tf + k1 * (1.0 - b + b * doc_len[doc_idx] / avgdl)
                scores[doc_idx] += term_weight * (tf * (k1 + 1.0) / denom)
        best = heapq.nlargest(top_k, scores.items(), key=lambda item: (item[1], evidence_ids[item[0]]))
        pool[cid] = [
            {'evidence_id': evidence_ids[doc_idx], 'score': float(score), 'rank': rank}
            for rank, (doc_idx, score) in enumerate(best, start=1)
        ]
        if i % 100 == 0:
            print(f'BM25 scoring pass: {i}/{len(claim_terms)} claims')
    print(f'BM25 finished in {time.perf_counter() - start:.1f}s')
    return pool


def trim_pool(pool, top_k):
    return {cid: rows[:top_k] for cid, rows in pool.items()}


def pool_union_ids(pool, claims, top_k):
    ids = []
    seen = set()
    for cid in claims:
        for row in pool.get(cid, [])[:top_k]:
            eid = row['evidence_id']
            if eid not in seen and eid in evidence:
                seen.add(eid)
                ids.append(eid)
    return ids

train_bm25_pool = build_query_bm25_pool(train_claims, evidence, min(BM25_TOP_K, len(evidence)))
target_bm25_pool = build_query_bm25_pool(target_claims, evidence, min(BM25_TOP_K, len(evidence)))
write_json(OUTPUT_DIR / 'train_bm25_pool.json', train_bm25_pool)
write_json(OUTPUT_DIR / 'target_bm25_pool.json', target_bm25_pool)
print_retrieval_metrics('target_bm25_top3', target_claims, target_bm25_pool, top_k=3)
print_retrieval_metrics('target_bm25_top500', target_claims, target_bm25_pool, top_k=min(500, BM25_TOP_K))

BM25 query vocab: 13812 terms
BM25 index pass: 100000/1208827
BM25 index pass: 200000/1208827
BM25 index pass: 300000/1208827
BM25 index pass: 400000/1208827
BM25 index pass: 500000/1208827
BM25 index pass: 600000/1208827
BM25 index pass: 700000/1208827
BM25 index pass: 800000/1208827
BM25 index pass: 900000/1208827
BM25 index pass: 1000000/1208827
BM25 index pass: 1100000/1208827
BM25 index pass: 1200000/1208827
BM25 postings kept: 8421/8421 terms
BM25 scoring pass: 100/1228 claims
BM25 scoring pass: 200/1228 claims
BM25 scoring pass: 300/1228 claims
BM25 scoring pass: 400/1228 claims
BM25 scoring pass: 500/1228 claims
BM25 scoring pass: 600/1228 claims
BM25 scoring pass: 700/1228 claims
BM25 scoring pass: 800/1228 claims
BM25 scoring pass: 900/1228 claims
BM25 scoring pass: 1000/1228 claims
BM25 scoring pass: 1100/1228 claims
BM25 scoring pass: 1200/1228 claims
BM25 finished in 64.6s
BM25 query vocab: 2610 terms
BM25 index pass: 100000/1208827
BM25 index pass: 200000/1208827
BM25 ind

{'evidence_f': 0.007320638187903192,
 'accuracy': 1.0,
 'harmonic': 0.01453487183797304,
 'macro_recall': 0.6090909090909091}

In [7]:
# BGE-small dense reranking inside the BM25 candidate pool.
def cls_pool(outputs):
    return outputs.last_hidden_state[:, 0]


@torch.no_grad()
def encode_texts(texts, tokenizer, model, batch_size=64, max_length=128, device=DEVICE, dtype=torch.float16):
    vectors = []
    model.eval()
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        batch = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
            out = model(**batch)
            emb = cls_pool(out)
        emb = F.normalize(emb.float(), dim=1).detach().cpu().to(dtype)
        vectors.append(emb)
        if start and start % (batch_size * 50) == 0:
            print(f'encoded {start}/{len(texts)} texts')
    return torch.cat(vectors, dim=0) if vectors else torch.empty((0, 0), dtype=dtype)


def dense_rerank_inside_pool(claims, evidence, prefilter_pool, model_name, prefilter_top_k, output_top_k, query_prefix='Represent this sentence for searching relevant passages: '):
    start = time.perf_counter()
    subset_ids = pool_union_ids(prefilter_pool, claims, prefilter_top_k)
    print(f'Dense subset union: {len(subset_ids)}/{len(evidence)} evidence rows')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)

    evidence_texts = [evidence[eid] for eid in subset_ids]
    evidence_emb = encode_texts(evidence_texts, tokenizer, model, batch_size=64, max_length=128, dtype=torch.float16)
    id_to_row = {eid: i for i, eid in enumerate(subset_ids)}

    claim_ids = list(claims.keys())
    claim_texts = [query_prefix + claims[cid]['claim_text'] for cid in claim_ids]
    claim_emb = encode_texts(claim_texts, tokenizer, model, batch_size=32, max_length=128, dtype=torch.float16)

    ranked = {}
    for claim_idx, cid in enumerate(claim_ids):
        cand_ids = [row['evidence_id'] for row in prefilter_pool.get(cid, [])[:prefilter_top_k] if row['evidence_id'] in id_to_row]
        if not cand_ids:
            ranked[cid] = []
            continue
        rows = torch.tensor([id_to_row[eid] for eid in cand_ids], dtype=torch.long)
        cand_emb = evidence_emb[rows].to(DEVICE).float()
        q = claim_emb[claim_idx].to(DEVICE).float()
        scores = (cand_emb @ q).detach().cpu().numpy()
        order = np.argsort(-scores)[:output_top_k]
        ranked[cid] = [
            {'evidence_id': cand_ids[j], 'score': float(scores[j]), 'rank': rank}
            for rank, j in enumerate(order, start=1)
        ]
        if (claim_idx + 1) % 100 == 0:
            print(f'dense rerank claims: {claim_idx + 1}/{len(claim_ids)}')

    del model, tokenizer, evidence_emb, claim_emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'Dense rerank finished in {time.perf_counter() - start:.1f}s')
    return ranked

target_dense_pool = dense_rerank_inside_pool(
    target_claims,
    evidence,
    target_bm25_pool,
    DENSE_MODEL_NAME,
    prefilter_top_k=min(DENSE_PREFILTER_K, BM25_TOP_K),
    output_top_k=DENSE_TOP_K,
)
write_json(OUTPUT_DIR / 'target_bge_small_ranked.json', target_dense_pool)
print_retrieval_metrics('target_dense_top3', target_claims, target_dense_pool, top_k=3)
print_retrieval_metrics('target_dense_top100', target_claims, target_dense_pool, top_k=min(100, DENSE_TOP_K))

Dense subset union: 154163/1208827 evidence rows


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoded 3200/154163 texts
encoded 6400/154163 texts
encoded 9600/154163 texts
encoded 12800/154163 texts
encoded 16000/154163 texts
encoded 19200/154163 texts
encoded 22400/154163 texts
encoded 25600/154163 texts
encoded 28800/154163 texts
encoded 32000/154163 texts
encoded 35200/154163 texts
encoded 38400/154163 texts
encoded 41600/154163 texts
encoded 44800/154163 texts
encoded 48000/154163 texts
encoded 51200/154163 texts
encoded 54400/154163 texts
encoded 57600/154163 texts
encoded 60800/154163 texts
encoded 64000/154163 texts
encoded 67200/154163 texts
encoded 70400/154163 texts
encoded 73600/154163 texts
encoded 76800/154163 texts
encoded 80000/154163 texts
encoded 83200/154163 texts
encoded 86400/154163 texts
encoded 89600/154163 texts
encoded 92800/154163 texts
encoded 96000/154163 texts
encoded 99200/154163 texts
encoded 102400/154163 texts
encoded 105600/154163 texts
encoded 108800/154163 texts
encoded 112000/154163 texts
encoded 115200/154163 texts
encoded 118400/154163 text

{'evidence_f': 0.034045287442477326,
 'accuracy': 1.0,
 'harmonic': 0.06584873574866754,
 'macro_recall': 0.5858225108225108}

In [8]:
# MiniLM binary evidence selector for top3 reranking.
class PairDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def collate_pairs(batch, tokenizer, max_length):
    encoded = tokenizer(
        [row['claim_text'] for row in batch],
        [row['evidence_text'] for row in batch],
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )
    encoded['labels'] = torch.tensor([row['label'] for row in batch], dtype=torch.long)
    return encoded


def make_binary_train_rows(claims, evidence, candidate_pool, source_k, max_negatives, seed=SEED):
    rng = random.Random(seed)
    rows = []
    positives = negatives = 0
    for cid, claim in claims.items():
        gold = set(claim.get('evidences', []))
        candidates = [row['evidence_id'] for row in candidate_pool.get(cid, [])[:source_k]]
        negs = [eid for eid in candidates if eid not in gold and eid in evidence]
        if len(negs) > max_negatives:
            head = negs[: max_negatives // 2]
            tail = negs[max_negatives // 2:]
            negs = head + rng.sample(tail, min(len(tail), max_negatives - len(head)))
        for eid in claim.get('evidences', []):
            if eid in evidence:
                rows.append({'claim_id': cid, 'evidence_id': eid, 'claim_text': claim['claim_text'], 'evidence_text': evidence[eid], 'label': 1})
                positives += 1
        for eid in negs:
            rows.append({'claim_id': cid, 'evidence_id': eid, 'claim_text': claim['claim_text'], 'evidence_text': evidence[eid], 'label': 0})
            negatives += 1
    rng.shuffle(rows)
    print({'binary_train_rows': len(rows), 'positives': positives, 'negatives': negatives})
    return rows


def train_binary_selector(train_rows):
    tokenizer = AutoTokenizer.from_pretrained(BINARY_SELECTOR_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        BINARY_SELECTOR_MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    loader = DataLoader(
        PairDataset(train_rows),
        batch_size=BINARY_BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda batch: collate_pairs(batch, tokenizer, BINARY_MAX_LENGTH),
    )
    for epoch in range(1, BINARY_EPOCHS + 1):
        model.train()
        total = 0.0
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch, labels=labels)
            outputs.loss.backward()
            optimizer.step()
            total += float(outputs.loss.detach().cpu())
        print({'binary_epoch': epoch, 'loss': total / max(1, len(loader))})
    return tokenizer, model


@torch.no_grad()
def score_binary_selector(claims, evidence, candidate_pool, tokenizer, model, source_k):
    pairs = []
    for cid, claim in claims.items():
        for row in candidate_pool.get(cid, [])[:source_k]:
            eid = row['evidence_id']
            if eid in evidence:
                pairs.append({'claim_id': cid, 'evidence_id': eid, 'claim_text': claim['claim_text'], 'evidence_text': evidence[eid], 'label': 0})
    loader = DataLoader(
        PairDataset(pairs),
        batch_size=BINARY_BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda batch: collate_pairs(batch, tokenizer, BINARY_MAX_LENGTH),
    )
    ranked = defaultdict(list)
    offset = 0
    model.eval()
    for batch in loader:
        _labels = batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        if logits.shape[-1] == 1:
            probs = torch.sigmoid(logits[:, 0])
        else:
            probs = torch.softmax(logits, dim=-1)[:, 1]
        probs = probs.detach().cpu().tolist()
        chunk = pairs[offset: offset + len(probs)]
        offset += len(probs)
        for row, prob in zip(chunk, probs):
            ranked[row['claim_id']].append({'evidence_id': row['evidence_id'], 'score': float(prob)})
    final = {}
    for cid, rows in ranked.items():
        rows.sort(key=lambda r: (-r['score'], r['evidence_id']))
        for rank, row in enumerate(rows, start=1):
            row['rank'] = rank
        final[cid] = rows
    return final


def rrf_fuse(claims, sources, weights, rrf_k=20.0, cap=80):
    fused = {}
    for cid in claims:
        scores = defaultdict(float)
        for name, source in sources.items():
            weight = weights.get(name, 0.0)
            if weight <= 0:
                continue
            for idx, row in enumerate(source.get(cid, [])[:cap], start=1):
                rank = int(row.get('rank', idx))
                scores[row['evidence_id']] += weight / (rrf_k + rank)
        rows = [{'evidence_id': eid, 'score': float(score)} for eid, score in scores.items()]
        rows.sort(key=lambda r: (-r['score'], r['evidence_id']))
        for rank, row in enumerate(rows, start=1):
            row['rank'] = rank
        fused[cid] = rows
    return fused

binary_train_rows = make_binary_train_rows(
    train_claims,
    evidence,
    train_bm25_pool,
    source_k=BINARY_TRAIN_SOURCE_K,
    max_negatives=BINARY_MAX_NEGATIVES,
)
binary_tokenizer, binary_model = train_binary_selector(binary_train_rows)

target_binary_ranked = score_binary_selector(
    target_claims,
    evidence,
    target_dense_pool,
    binary_tokenizer,
    binary_model,
    source_k=min(BINARY_SCORE_TOP_K, DENSE_TOP_K),
)
write_json(OUTPUT_DIR / 'target_binary_selector_ranked.json', target_binary_ranked)
print_retrieval_metrics('target_binary_top3', target_claims, target_binary_ranked, top_k=3)

final_top3_ranked = rrf_fuse(
    target_claims,
    sources={'binary': target_binary_ranked, 'dense': target_dense_pool, 'bm25': target_bm25_pool},
    weights={'binary': 2.0, 'dense': 0.5, 'bm25': 0.25},
    rrf_k=20.0,
    cap=min(BINARY_SCORE_TOP_K, DENSE_TOP_K),
)
write_json(OUTPUT_DIR / 'final_top3_ranked.json', final_top3_ranked)
print_retrieval_metrics('final_top3_rrf', target_claims, final_top3_ranked, top_k=FINAL_EVIDENCE_TOP_K)

# Free the selector before classifier training if GPU memory is tight.
del binary_model, binary_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

{'binary_train_rows': 40007, 'positives': 4122, 'negatives': 35885}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1`.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([2, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


{'binary_epoch': 1, 'loss': 0.255054236773511}
{'binary_epoch': 2, 'loss': 0.1571810525357634}
target_binary_top3 {
  "evidence_f": 0.06889301175015462,
  "accuracy": 1.0,
  "harmonic": 0.12890534598472578,
  "macro_recall": 0.07965367965367964
}
final_top3_rrf {
  "evidence_f": 0.12714904143475572,
  "accuracy": 1.0,
  "harmonic": 0.22561176341490177,
  "macro_recall": 0.1536796536796537
}


33

In [9]:
# DistilRoBERTa claim classifier trained in-notebook.
def row_to_classifier_text(claim, context_rows, evidence, top_k=10, evidence_token_budget=45):
    parts = [f"CLAIM: {claim['claim_text']}"]
    for row in context_rows[:top_k]:
        eid = row['evidence_id']
        text = ' '.join(evidence.get(eid, '').split()[:evidence_token_budget])
        parts.append(f'EVIDENCE: {text}')
    return '\n'.join(parts)


class ClaimDataset(Dataset):
    def __init__(self, claims, context_pool, evidence, require_label=True):
        self.items = []
        for cid, claim in claims.items():
            if require_label and 'claim_label' not in claim:
                continue
            self.items.append((cid, claim, context_pool.get(cid, [])))
        self.evidence = evidence
        self.require_label = require_label

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        cid, claim, context = self.items[idx]
        text = row_to_classifier_text(claim, context, self.evidence, CLASSIFIER_CONTEXT_TOP_K, EVIDENCE_TOKEN_BUDGET)
        label = LABEL_TO_ID.get(claim.get('claim_label', 'SUPPORTS'), 0)
        return {'claim_id': cid, 'text': text, 'label': label}


def collate_claims(batch, tokenizer, max_length, with_labels=True):
    encoded = tokenizer(
        [row['text'] for row in batch],
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )
    if with_labels:
        encoded['labels'] = torch.tensor([row['label'] for row in batch], dtype=torch.long)
    return encoded


def evaluate_classifier(model, tokenizer, dataset):
    loader = DataLoader(dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_claims(b, tokenizer, CLASSIFIER_MAX_LENGTH, True))
    labels, preds = [], []
    total_loss = 0.0
    model.eval()
    with torch.no_grad():
        for batch in loader:
            gold = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch, labels=gold)
            total_loss += float(outputs.loss.detach().cpu())
            pred = torch.argmax(outputs.logits, dim=-1)
            labels.extend(gold.detach().cpu().tolist())
            preds.extend(pred.detach().cpu().tolist())
    return {
        'loss': total_loss / max(1, len(loader)),
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, labels=list(range(len(LABELS))), average='macro'),
        'labels': labels,
        'predictions': preds,
    }


def train_claim_classifier(train_claims, target_claims, train_context_pool, target_context_pool):
    tokenizer = AutoTokenizer.from_pretrained(CLAIM_CLASSIFIER_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        CLAIM_CLASSIFIER_MODEL_NAME,
        num_labels=len(LABELS),
        id2label={i: label for i, label in enumerate(LABELS)},
        label2id=LABEL_TO_ID,
    ).to(DEVICE)
    train_ds = ClaimDataset(train_claims, train_context_pool, evidence, require_label=True)
    target_has_labels = all('claim_label' in c for c in target_claims.values())
    target_ds = ClaimDataset(target_claims, target_context_pool, evidence, require_label=target_has_labels)
    loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_claims(b, tokenizer, CLASSIFIER_MAX_LENGTH, True))
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    best_state = None
    best_eval = None
    curve = []
    for epoch in range(1, CLASSIFIER_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch, labels=labels)
            outputs.loss.backward()
            optimizer.step()
            total_loss += float(outputs.loss.detach().cpu())
        row = {'epoch': epoch, 'train_loss': total_loss / max(1, len(loader))}
        if target_has_labels:
            dev_eval = evaluate_classifier(model, tokenizer, target_ds)
            row.update({'target_accuracy': dev_eval['accuracy'], 'target_macro_f1': dev_eval['macro_f1'], 'target_loss': dev_eval['loss']})
            if best_eval is None or (dev_eval['accuracy'], dev_eval['macro_f1'], -dev_eval['loss']) > (best_eval['accuracy'], best_eval['macro_f1'], -best_eval['loss']):
                best_eval = dev_eval
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(row)
        curve.append(row)
    if best_state is not None:
        model.load_state_dict(best_state)
    return tokenizer, model, curve


@torch.no_grad()
def predict_claim_labels(claims, context_pool, tokenizer, model):
    ds = ClaimDataset(claims, context_pool, evidence, require_label=False)
    loader = DataLoader(ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False, collate_fn=lambda b: b)
    labels_by_claim = {}
    model.eval()
    for batch_rows in loader:
        encoded = tokenizer([r['text'] for r in batch_rows], padding=True, truncation=True, max_length=CLASSIFIER_MAX_LENGTH, return_tensors='pt')
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        pred = torch.argmax(model(**encoded).logits, dim=-1).detach().cpu().tolist()
        for row, label_id in zip(batch_rows, pred):
            labels_by_claim[row['claim_id']] = ID_TO_LABEL[label_id]
    return labels_by_claim

# Use BM25 context for training and final top3/binary context for target classification.
train_classifier_context = train_bm25_pool
target_classifier_context = rrf_fuse(
    target_claims,
    sources={'binary': target_binary_ranked, 'dense': target_dense_pool, 'bm25': target_bm25_pool},
    weights={'binary': 1.5, 'dense': 1.0, 'bm25': 0.25},
    rrf_k=20.0,
    cap=max(CLASSIFIER_CONTEXT_TOP_K, BINARY_SCORE_TOP_K),
)
write_json(OUTPUT_DIR / 'target_classifier_context_ranked.json', target_classifier_context)

classifier_tokenizer, classifier_model, classifier_curve = train_claim_classifier(
    train_claims,
    target_claims,
    train_classifier_context,
    target_classifier_context,
)
labels_by_claim = predict_claim_labels(target_claims, target_classifier_context, classifier_tokenizer, classifier_model)
write_json(OUTPUT_DIR / 'predicted_labels.json', labels_by_claim)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'epoch': 1, 'train_loss': 1.2736955602447708, 'target_accuracy': 0.44155844155844154, 'target_macro_f1': 0.15315315315315314, 'target_loss': 1.2686249524354936}
{'epoch': 2, 'train_loss': 1.240530126667642, 'target_accuracy': 0.44155844155844154, 'target_macro_f1': 0.20332988267770877, 'target_loss': 1.2495660781860352}
{'epoch': 3, 'train_loss': 1.1965641952180244, 'target_accuracy': 0.43506493506493504, 'target_macro_f1': 0.16294329793628531, 'target_loss': 1.2150744795799255}
{'epoch': 4, 'train_loss': 1.1139049023002774, 'target_accuracy': 0.4155844155844156, 'target_macro_f1': 0.19815889130609943, 'target_loss': 1.3092229038476944}
{'epoch': 5, 'train_loss': 0.9354791157431417, 'target_accuracy': 0.474025974025974, 'target_macro_f1': 0.30313670411985016, 'target_loss': 1.3598127573728562}


# 3.Testing and Evaluation

In [10]:
# Final prediction file.
final_predictions = predictions_from_ranked(
    target_claims,
    final_top3_ranked,
    labels_by_claim=labels_by_claim,
    default_label=majority_label(train_claims),
    top_k=FINAL_EVIDENCE_TOP_K,
)
write_json(OUTPUT_DIR / 'final_predictions.json', final_predictions)
print('Wrote', OUTPUT_DIR / 'final_predictions.json')

summary = {
    'output_dir': str(OUTPUT_DIR),
    'device': str(DEVICE),
    'dense_model': DENSE_MODEL_NAME,
    'binary_selector_model': BINARY_SELECTOR_MODEL_NAME,
    'claim_classifier_model': CLAIM_CLASSIFIER_MODEL_NAME,
    'bm25_top_k': BM25_TOP_K,
    'dense_prefilter_k': DENSE_PREFILTER_K,
    'dense_top_k': DENSE_TOP_K,
    'binary_score_top_k': BINARY_SCORE_TOP_K,
    'classifier_context_top_k': CLASSIFIER_CONTEXT_TOP_K,
    'classifier_curve': classifier_curve,
}

if all('claim_label' in c for c in target_claims.values()):
    final_metrics = assignment_metrics(target_claims, final_predictions)
    final_metrics['top3_macro_recall'] = macro_recall(target_claims, final_predictions)
    y_true = [LABEL_TO_ID[target_claims[cid]['claim_label']] for cid in target_claims]
    y_pred = [LABEL_TO_ID[final_predictions[cid]['claim_label']] for cid in target_claims]
    final_metrics['classification_macro_f1'] = f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average='macro')
    summary['final_metrics'] = final_metrics
    print('FINAL METRICS')
    print(json.dumps(final_metrics, indent=2))
    print('CLASSIFICATION REPORT')
    print(classification_report(y_true, y_pred, target_names=LABELS, labels=list(range(len(LABELS))), zero_division=0))
    print('CONFUSION MATRIX')
    print(confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS)))))
else:
    print('Target claims are unlabelled; evaluation skipped.')

write_json(OUTPUT_DIR / 'run_summary.json', summary)
print('Wrote', OUTPUT_DIR / 'run_summary.json')

Wrote /mnt/a/code/NLP/A3/outputs/round16_colab_full_pipeline_local/final_predictions.json
FINAL METRICS
{
  "evidence_f": 0.12714904143475572,
  "accuracy": 0.474025974025974,
  "harmonic": 0.2005138159854753,
  "top3_macro_recall": 0.1536796536796537,
  "classification_macro_f1": 0.30313670411985016
}
CLASSIFICATION REPORT
                 precision    recall  f1-score   support

       SUPPORTS       0.51      0.82      0.63        68
        REFUTES       0.38      0.19      0.25        27
NOT_ENOUGH_INFO       0.39      0.29      0.33        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.47       154
      macro avg       0.32      0.33      0.30       154
   weighted avg       0.40      0.47      0.41       154

CONFUSION MATRIX
[[56  3  9  0]
 [15  5  7  0]
 [26  3 12  0]
 [13  2  3  0]]
Wrote /mnt/a/code/NLP/A3/outputs/round16_colab_full_pipeline_local/run_summary.json


In [11]:
# Optional: copy outputs to Google Drive after the run.
# This stores predictions and logs only. Do not include course data or model checkpoints in a final submission zip.
COPY_TO_DRIVE = False
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/COMP90042_A3_outputs')

if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['final_predictions.json', 'run_summary.json', 'target_classifier_context_ranked.json', 'final_top3_ranked.json']:
        src = OUTPUT_DIR / name
        if src.exists():
            dst = DRIVE_OUTPUT_DIR / name
            dst.write_text(src.read_text(encoding='utf-8'), encoding='utf-8')
            print('copied', src, '->', dst)

## Object Oriented Programming codes here

The implementation above uses lightweight `torch.utils.data.Dataset` classes for pair reranking and claim classification. The rest is functional so the notebook is easier to audit in Colab.